# AlphaZero Chess - Interactive Training Notebook

This notebook provides a simplified, step-by-step walkthrough of the AlphaZero training process. Unlike the main script (`main.py`) which uses parallel processing for high performance, this notebook runs a single, sequential loop. 

**Purpose:** This is for demonstration, debugging, and understanding the core components. It is **not** for serious training, as it will be very slow.

## 1. Setup and Initialization

First, we import our modules and initialize the core components: the neural network and the replay buffer. We will also load the latest model checkpoint if one exists.

In [ ]:
import os
import numpy as np

from . import config
from . import network
from . import replay_buffer
from . import self_play
from . import train

# --- Initialize Components ---
current_network = network.create_model()
buffer = replay_buffer.ReplayBuffer()

# --- Load latest checkpoint ---
if not os.path.exists(config.CHECKPOINT_DIR):
    os.makedirs(config.CHECKPOINT_DIR)

latest_model_path = os.path.join(config.CHECKPOINT_DIR, "latest_model.h5")
if os.path.exists(latest_model_path):
    print(f"Loading weights from {latest_model_path}")
    current_network.load_weights(latest_model_path)
else:
    print("Starting with a new, untrained model.")

## 2. The Core Loop: Self-Play and Train

Here, we'll run a single iteration of the main AlphaZero loop:
1.  **Self-Play**: Play one full game where the engine plays against itself, using MCTS to guide its moves.
2.  **Store Data**: Format the results of the game and save them to the replay buffer.
3.  **Train**: Sample a batch of data from the replay buffer and perform one training step on the neural network.

In [ ]:
# --- 1. Self-Play Phase ---
print("--- Starting Self-Play Game 1 ---")
# In a real scenario, this would be one of many parallel games.
# We pass the model path to simulate how the parallel workers would load it.
current_network.save_weights(latest_model_path) # Ensure the latest weights are on disk
game_history = self_play.run_game(model_path=latest_model_path)
print(f"Self-play game finished. Result: {game_history.outcome}")

# --- 2. Store Data Phase ---
formatted_history = self_play.format_game_history_for_buffer(game_history)
buffer.save_game(formatted_history)
print(f"Game data saved to replay buffer. Buffer size is now: {len(buffer)}")

# --- 3. Training Phase ---
print("\n--- Starting Training Step 1 ---")
if len(buffer) < config.BATCH_SIZE:
    print("Not enough data in buffer to train. Play more games first.")
else:
    loss_metrics = train.train_step(current_network, buffer)
    if loss_metrics:
        print(f"Training step complete.")
        print(f"  Total Loss: {loss_metrics['total_loss']:.4f}")
        print(f"  Policy Loss: {loss_metrics['policy_loss']:.4f}")
        print(f"  Value Loss: {loss_metrics['value_loss']:.4f}")

        # Save the newly trained weights
        current_network.save_weights(latest_model_path)
        print(f"\nUpdated model weights saved to {latest_model_path}")